<a href="https://colab.research.google.com/github/2yngsklab/EchoRNA/blob/main/tutorial/sample_EchoRNA_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [Tutorial](https://github.com/2yngsklab/EchoRNA): EchoRNA

## **Overview**

EchoRNA is a discrete diffusion model that generates functional RNA sequences conditioned on the 3D structure of an RNA-binding protein (RBP). In this tutorial, we use EchoRNA to generate protein-binding RNA sequences, then use [Boltz-2](https://github.com/jwohlwend/boltz) to predict the resulting protein–RNA complex structure.

**This tutorial requires a GPU session. Please change your runtime type to GPU.**

In [ ]:
### check for GPU availability
# If you have a CPU-only runtime, you'll see an error such as:
#   /bin/bash: line 1: nvidia-smi: command not found

!nvidia-smi

Thu Jul  9 10:30:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# **Environment Setup**

In [ ]:
### install Boltz-2 and visualization tool
!pip install boltz[cuda] -U
!pip install py3Dmol ipywidgets biopython -q

In [ ]:
### install pacakges for EchoRNA
# you might need to restart the session to downgrade numpy
!pip install torch==2.4.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric -f https://data.pyg.org/whl/torch-2.4.0+cu118.html
!pip install numpy pandas rna-fm pyyaml fair-esm==2.0.0 biotite==1.2 huggingface_hub

Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.7/857.7 MB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 88.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 63.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 123.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 14.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.9/142.9 MB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1

In [ ]:
### Patch a deprecated biotite call in fair-esm's inverse_folding/util.py
%%bash
echo "Patching deprecated function in fair-esm..."
python - <<'PYEOF'
import importlib.util, os, sys, re
spec = importlib.util.find_spec("esm")
if spec is None or spec.origin is None:
    sys.stderr.write("Could not locate esm package.\n"); sys.exit(1)
path = os.path.join(os.path.dirname(spec.origin), "inverse_folding", "util.py")
if not os.path.isfile(path):
    sys.stderr.write("Could not locate util.py at %s\n" % path); sys.exit(1)
with open(path, "r", encoding="utf-8") as f:
    text = f.read()
if re.search(r"\bfilter_backbone\b", text):
    text = re.sub(r"\bfilter_backbone\b", "filter_peptide_backbone", text)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
    print("Patched: %s" % path)
else:
    print("No 'filter_backbone' found in %s (already patched or upstream changed); skipping." % path)
PYEOF

Patching deprecated function in fair-esm...
Patched: /usr/local/lib/python3.12/dist-packages/esm/inverse_folding/util.py


# **Install EchoRNA**

In [ ]:
### clone EchoRNA source code from git repository
!git clone https://github.com/2yngsklab/EchoRNA.git

Cloning into 'EchoRNA'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 45 (delta 3), reused 31 (delta 1), pack-reused 9 (from 2)
Receiving objects: 100% (45/45), 7.05 MiB | 10.60 MiB/s, done.
Resolving deltas: 100% (3/3), done.


In [ ]:
### download EchoRNA model weights
!hf download 2yngsklab/EchoRNA echorna_weight.pth --local-dir /content/EchoRNA

Hint: A new version of huggingface_hub (1.22.0) is available! You are using version 1.20.1.
To update, run: hf update
echorna_weight.pth: 100% 656M/656M [00:07<00:00, 92.1MB/s]
✓ Downloaded
  path: /content/EchoRNA/echorna_weight.pth


In [ ]:
### download other pretrained model weights (RNA-FM, ESM-2, ESM-IF)
import fm, esm

esm.pretrained.esm2_t33_650M_UR50D() # ESM-2
esm.pretrained.esm_if1_gvp4_t16_142M_UR50() # ESM-IF
fm.pretrained.rna_fm_t12() # RNA-FM

Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t33_650M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t33_650M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D-contact-regression.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm_if1_gvp4_t16_142M_UR50.pt" to /root/.cache/torch/hub/checkpoints/esm_if1_gvp4_t16_142M_UR50.pt
/usr/local/lib/python3.12/dist-packages/esm/pretrained.py:215: UserWarning: Regression weights not found, predicting contacts will not produce correct results.
  warnings.warn(


(GVPTransformerModel(
   (encoder): GVPTransformerEncoder(
     (dropout_module): Dropout(p=0.1, inplace=False)
     (embed_tokens): Embedding(35, 512, padding_idx=1)
     (embed_positions): SinusoidalPositionalEmbedding()
     (embed_gvp_input_features): Linear(in_features=15, out_features=512, bias=True)
     (embed_confidence): Linear(in_features=16, out_features=512, bias=True)
     (embed_dihedrals): DihedralFeatures(
       (node_embedding): Linear(in_features=6, out_features=512, bias=True)
       (norm_nodes): Normalize()
     )
     (gvp_encoder): GVPEncoder(
       (embed_graph): GVPGraphEmbedding(
         (embed_node): Sequential(
           (0): GVP(
             (wh): Linear(in_features=3, out_features=256, bias=False)
             (ws): Linear(in_features=263, out_features=1024, bias=True)
             (wv): Linear(in_features=256, out_features=256, bias=False)
           )
           (1): LayerNorm(
             (scalar_norm): LayerNorm((1024,), eps=1e-05, elementwise_a

In [ ]:
### Uncomment followings if the RNA-FM download fails
# !mkdir -p /root/.cache/torch/hub/checkpoints/
# !hf download cuhkaih/rnafm RNA-FM_pretrained.pth --local-dir /root/.cache/torch/hub/checkpoints/

RNA-FM_pretrained.pth: 100% 1.19G/1.19G [00:10<00:00, 110MB/s]
✓ Downloaded
  path: /root/.cache/torch/hub/checkpoints/RNA-FM_pretrained.pth


# **EchoRNA Sampling Parameters**

You can control the sampling of EchoRNAs with following parameters. Please go to our [Github tutorial](https://github.com/2yngsklab/EchoRNA#usage) for more information.

* **-p, --protein** : Input structure file in CIF format.
* **-c, --chain** : Protein chain ID to use.
* **-d, --output-dir** : Directory where the generated sequences are saved (default: `./output`).
* **-fn, --name** : Name of the output file (default: name of the CIF file).
* **-n, --num-sequence** : Number of RNA sequences to generate (default: `100`).
* **-l, --rna-length** : Length of RNA sequences to generate (default: `20`).
* **-s, --sampling-strategy** : Sampling strategy; choose between `vanilla` and `gumbel` (default: `vanilla`).
* **-sd, --random-seed** : Random seed (default: `42`).
* **-g, --GPU** : Enable GPU usage. Optionally specify a device using `cuda:<number>` (default: `cuda:0`). If not provided, CPU will be used.
* **--config** : Model configuration file (default: `./echorna_config.yaml`).
* **--weight** : Model weight file (default: `./echorna_weight.pth`).

# **Example 1: Generate PUM2 ([3Q0Q](https://www.rcsb.org/structure/3Q0Q)) binding RNA sequences**

PUM2 (Pumilio RNA Binding Family Member 2) is an RNA-binding protein that regulates post-transcriptional gene expression by binding to the 3' UTR of target mRNAs, thereby controlling their stability and translation.

In this example, we generate PUM2-binding EchoRNAs using `3Q0Q.cif` in `./tutorial` and predict the protein–RNA complex with [Boltz-2](https://github.com/jwohlwend/boltz).



In [ ]:
### move to the working directory
%cd /content/EchoRNA

/content/EchoRNA


In [ ]:
### put your parameters for EchoRNA input
protein_name = '3Q0Q'
protein_cif = '/content/EchoRNA/tutorial/3Q0Q.cif'
chain = 'A'

num_sequence = 100
length = 8

file_name = f'{protein_name}_{num_sequence}_{length}'

In [ ]:
### generate 100 EchoRNA sequences of length 8 for the PUM2 protein
# If EchoRNA ran correctly, you should see a confirmation message like:
#   Wrote 100 RNA sequences to /content/EchoRNA/output/RNA/3Q0Q_100_8.fasta
!python sample_EchoRNA.py -p {protein_cif} -c {chain} -n {num_sequence} -l {length} -fn {file_name} -g

Device: cuda:0
Parsing protein 3Q0Q.cif, chain A
Loading ESM models and generating embeddings
/usr/local/lib/python3.12/dist-packages/esm/pretrained.py:215: UserWarning: Regression weights not found, predicting contacts will not produce correct results.
  warnings.warn(
Constructing and saving protein data
  Saved: /content/EchoRNA/output/complex/3Q0Q_A.pickle
  Saved: /content/EchoRNA/output/esm2/3Q0Q_A.pt
  Saved: /content/EchoRNA/output/esmIF/3Q0Q_A.pt
Loading diffusion model and generating 100 RNA sequences of length 8
Wrote 100 RNA sequences to /content/EchoRNA/output/RNA/3Q0Q_100_8.fasta


In [ ]:
### List first 10 EchoRNA sequences
from Bio import SeqIO

fasta_path = f"/content/EchoRNA/output/RNA/{file_name}.fasta"

records = list(SeqIO.parse(fasta_path, "fasta"))
print(f"First 10 sequences generated for PUM2\n")

for record in records[:10]:
    print(f"{record.seq}")

First 10 sequences generated for PUM2

UGUACCUU
UGUACUUG
UGUGAAUG
UGUGCCUU
UGUACUUU
UGUACUUG
UGUACCUU
UGCUUUGC
UGUGCCUU
UGUGAAUG


## **Visualize the Protein–RNA Complex**

1. In the PDB complex, the protein chain is colored green, and the RNA strand is colored blue.
2. In the predicted complex, residues are colored by pLDDT score.

In [ ]:
### create an input for Boltz-2 prediction
import yaml
import os
from Bio.PDB import MMCIFParser
from Bio.PDB.Polypeptide import PPBuilder


# Protein sequence
parser = MMCIFParser(QUIET=True)

structure_id = os.path.splitext(os.path.basename(protein_cif))[0]
structure = parser.get_structure(structure_id, protein_cif)

ppb = PPBuilder()
protein_seq = "".join(str(pp.get_sequence()) for pp in ppb.build_peptides(structure[0][chain]))

# RNA sequence
rna_seq = str(records[0].seq)

file_name_complex = f'{protein_name}_{rna_seq}'

input_dict = {
    "version": 1,
    "sequences": [
        {"protein": {"id": "A", "sequence": protein_seq}},
        {"rna": {"id": "B", "sequence": rna_seq}},
    ],
}

os.makedirs("/content/boltz_run", exist_ok=True)
yaml_path = f"/content/boltz_run/{file_name_complex}.yaml"

with open(yaml_path, "w") as f:
    yaml.dump(input_dict, f, sort_keys=False, default_flow_style=False)

print(open(yaml_path).read())

version: 1
sequences:
- protein:
    id: A
    sequence: GRSRLLEDFRNNRFPNLQLRDLIGHIVEFSQDQHGSRFIQQKLERATPAERQMVFNEILQAAYQLMTDVFGNYVIQKFFEFGSLDQKLALATRIRGHVLPLALQMYGCRVIQKALESISSDQQVISEMVKELDGHVLKCVKDQNGNHVVQKCIECVQPQSLQFIIDAFKGQVFVLSTHPYGCRVIQRILEHCTAEQTLPILEELHQHTEQLVQDQYGNYVIQHVLEHGRPEDKSKIVSEIRGKVLALSQHKFASNVVEKCVTHASRAERALLIDEVCCQNDGPHSALYTMMKDQYANYVVQKMIDMAEPAQRKIIMHKIRPHITTLRKYTYGKHILAKLEKYY
- rna:
    id: B
    sequence: UGUACCUU



In [ ]:
### Boltz-2 prediction
# If the Boltz-2 prediction succeeds, you should see a summary message like:
#   Number of failed examples: 0
!boltz predict {yaml_path} \
    --use_msa_server \
    --out_dir /content/boltz_run/results \
    --output_format pdb

MSA server enabled: https://api.colabfold.com
MSA server authentication: no credentials provided
Extracting the CCD data to /root/.boltz/mols. This may take a bit of time. You may change the cache directory with the --cache flag.
Checking input data.
Processing 1 inputs with 1 threads.
  0% 0/1 [00:00<?, ?it/s]Generating MSA for /content/boltz_run/3Q0Q_UGUACCUU.yaml with 1 protein entities.
Calling MSA server for target 3Q0Q_UGUACCUU with 1 sequences
MSA server URL: https://api.colabfold.com
MSA pairing strategy: greedy
No authentication provided for MSA server

  0% 0/150 [00:00<?, ?it/s]
SUBMIT:   0% 0/150 [00:00<?, ?it/s]
COMPLETE:   0% 0/150 [00:00<?, ?it/s]
COMPLETE: 100% 150/150 [00:00<00:00, 190.86it/s]
100% 1/1 [00:01<00:00,  1.72s/it]
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Running structure prediction for 1 input.
/usr/local/lib/python3.12/dist-packages/p

In [ ]:
import glob
import json

# Check confidence scores
conf_files = glob.glob(f"/content/boltz_run/results/boltz_results_3Q0Q_EchoRNA/predictions/{file_name_complex}/confidence_{file_name_complex}_model_0.json", recursive=True)
for cf in conf_files:
    with open(cf) as fh:
        data = json.load(fh)
    print(" pLDDT:", data.get("complex_plddt"))
    print(" ipTM:", data.get("iptm"))
    print(" pTM:", data.get("ptm"))

In [ ]:
import glob
import py3Dmol
import ipywidgets as widgets
from IPython.display import display
from Bio.PDB import PDBParser


def extract_residue_plddt(pdb_path):
    """
    Extract per-residue pLDDT scores grouped by chain.
    Boltz stores pLDDT (x100) in the B-factor column; all atoms in a residue
    carry the same value, so we just take the first atom's B-factor.
    """
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("model", pdb_path)

    plddt_by_chain = {}
    for chain in structure[0]:
        scores = []
        for res in chain:
            atoms = list(res.get_atoms())
            if atoms:
                scores.append(atoms[0].get_bfactor())
        plddt_by_chain[chain.id] = scores
    return plddt_by_chain


def apply_plddt_style(view, plddt_by_chain, add_hetflag=False):
    """Color cartoon per-residue according to pLDDT, iterating chain by chain."""
    view.setStyle({}, {})  # clear any previous style

    for chain_id, plddt in plddt_by_chain.items():
        for i, s in enumerate(plddt):
            view.setStyle(
                {"chain": chain_id, "resi": i + 1},
                {"cartoon": {"color": plddt_hex(s)}},
            )

    if add_hetflag:
        view.addStyle({"hetflag": True}, {"stick": {}})

    view.zoomTo()

def plddt_hex(v):
    """Convert pLDDT score to hex color, matching the 'roygb' gradient (red=low -> blue=high)."""
    if v >= 90:
        return "#0000FF"  # blue (high confidence)
    if v >= 70:
        return "#9ACD32"  # yellow-green
    if v >= 50:
        return "#FFA500"  # orange
    return "#FF0000"      # red (low confidence)


PLDDT_LEGEND = (
    '<div style="text-align:center; font-weight:bold; font-size:12px; margin-bottom:2px;">pLDDT score</div>'
    '<div style="text-align:center;">'
    '<span style="color:#FF0000;">&#9632;</span> &lt;50 &nbsp;'
    '<span style="color:#FFA500;">&#9632;</span> 50–70 &nbsp;'
    '<span style="color:#9ACD32;">&#9632;</span> 70–90 &nbsp;'
    '<span style="color:#0000FF;">&#9632;</span> &gt;90'
    '</div>'
)

CHAIN_LEGEND = (
    '<div style="text-align:center;">'
    '<div style="display:inline-block; text-align:left;">'
    '<div><span style="color:green;">&#9632;</span> Protein</div>'
    '<div><span style="color:blue;">&#9632;</span> RNA</div>'
    '</div>'
    '</div>'
)

In [ ]:
### Visualize the Protein–RNA Complex
# Load the original PDB (CIF) structure
with open(protein_cif) as f:
    cif_data = f.read()

# Load the predicted structure
result_files = glob.glob(f"/content/boltz_run/results/boltz_results_{file_name_complex}/predictions/{file_name_complex}/{file_name_complex}_model_0.pdb", recursive=True)
pdb_path = result_files[0]
with open(pdb_path) as f:
    pdb_data = f.read()

plddt_by_chain = extract_residue_plddt(pdb_path)

# Left viewer: original PDB structure, colored by chain
view_left = py3Dmol.view(width=480, height=450)
view_left.addModel(cif_data, "cif")
view_left.setStyle({"chain": "A"}, {"cartoon": {"color": "green"}})  # protein
view_left.setStyle({"chain": "B"}, {"cartoon": {"color": "blue"}})   # RNA
view_left.zoomTo()


# Right viewer: predicted complex, colored by pLDDT, per residue, per chain
view_right = py3Dmol.view(width=480, height=450)
view_right.addModel(pdb_data, "pdb")
apply_plddt_style(view_right, plddt_by_chain)

# Capture each viewer into its own output widget
left_output = widgets.Output()
with left_output:
    view_left.show()

right_output = widgets.Output()
with right_output:
    view_right.show()

left_title = widgets.HTML('<div style="text-align:center; font-weight:bold;">PDB Structure (3Q0Q)</div>')
right_title = widgets.HTML('<div style="text-align:center; font-weight:bold;">Predicted Structure</div>')
chain_legend_widget = widgets.HTML(value=CHAIN_LEGEND)
plddt_legend_widget = widgets.HTML(value=PLDDT_LEGEND)

left_col = widgets.VBox([left_title, left_output, chain_legend_widget])
right_col = widgets.VBox([right_title, right_output, plddt_legend_widget])

display(widgets.HBox([left_col, right_col]))